# Гео-канал

У запроса есть `search_location_id` (числовой ID зоны, 2782 уникальных значения), у объявления - `item_location_id` + реальные `item_latitude`/`item_longitude`. Разберёмся, как это лучше использовать.

In [1]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd

from eval import make_val_split, recall_at_50
from geo import build_geo_tree, build_loc_centroid, haversine

train = pd.read_parquet("../data/train.parquet")
train_part, val_queries, val_relevant = make_val_split(train, val_size=0.1, random_state=42)
print("train_part:", len(train_part), "val_queries:", len(val_queries))

train_part: 448040 val_queries: 35447


## Локация не бесполезна

`search_location_id == item_location_id` совпадает точно только в **83.1%** случаев (много выездных/удалённых услуг) - НЕ хардфильтр, отдельный сигнал.

In [2]:
(train["search_location_id"] == train["item_location_id"]).mean()

np.float64(0.8310195650557696)

## Локация - это зона, не точка

`location_id` - это район/город, а не точка, и точное совпадение ID теряет информацию о том, НАСКОЛЬКО далеко объявление, если ID не совпал.

In [3]:
items = train.drop_duplicates("item_id").copy()
items["item_latitude"] = items["item_latitude"].astype(float)
items["item_longitude"] = items["item_longitude"].astype(float)

coord_std = items.groupby("item_location_id")[["item_latitude", "item_longitude"]].apply(lambda g: g.std().max())
print("медианный std координат внутри location_id в градусах):", coord_std.median())
print("в км:", coord_std.median() * 111)

медианный std координат внутри location_id в градусах): 0.02004449638661928
в км: 2.22493909891474


## Дистанция вместо ID: у запроса нет своих координат

Запрос даёт только `search_location_id`.\
 Аппроксимируем координаты зоны как **медиану** координат объявлений в `train_part` с тем же `item_location_id` - центроид зоны, без утечки из val.

In [4]:
loc_centroid = build_loc_centroid(train_part)
print("центроидов:", len(loc_centroid))
coverage = val_queries["search_location_id"].isin(loc_centroid.index).mean()
print("доля val_queries с центроидом:", coverage)

центроидов: 2589
доля val_queries с центроидом: 0.8757017519113042


Проверим сигнал напрямую: дистанция до истинного item должна быть намного меньше, чем до случайного.

In [5]:
sample = train.sample(20000, random_state=1).reset_index(drop=True)
sample["item_latitude"] = sample["item_latitude"].astype(float)
sample["item_longitude"] = sample["item_longitude"].astype(float)
centroid = loc_centroid.reindex(sample["search_location_id"]).reset_index(drop=True)
mask = centroid["item_latitude"].notna()
sample, centroid = sample[mask].reset_index(drop=True), centroid[mask].reset_index(drop=True)

dist_true = haversine(centroid["item_latitude"], centroid["item_longitude"], sample["item_latitude"], sample["item_longitude"])
random_items = train.sample(len(sample), random_state=2)[["item_latitude", "item_longitude"]].astype(float).reset_index(drop=True)
dist_random = haversine(centroid["item_latitude"], centroid["item_longitude"], random_items["item_latitude"], random_items["item_longitude"])

print("медиана дистанции до истинного item (км):", dist_true.median())
print("медиана дистанции до случайного item (км):", dist_random.median())
print("доля истинных item в радиусе 5км:", (dist_true <= 5).mean())
print("доля случайных item в радиусе 5км:", (dist_random <= 5).mean())

медиана дистанции до истинного item (км): 2.881070885921557
медиана дистанции до случайного item (км): 1211.7358605718555
доля истинных item в радиусе 5км: 0.662908744180829
доля случайных item в радиусе 5км: 0.006842784227943238


## top50 ближайших по BallTree (haversine)

In [6]:
geo_tree, geo_item_ids = build_geo_tree(train.drop_duplicates("item_id")[["item_id", "item_latitude", "item_longitude"]])

query_coords = loc_centroid.reindex(val_queries["search_location_id"])[["item_latitude", "item_longitude"]].to_numpy()
valid_mask = ~np.isnan(query_coords).any(axis=1)
_, indices = geo_tree.query(np.radians(query_coords[valid_mask]), k=50)

geo_dist_predictions = {}
valid_context_ids = val_queries["context_id"].to_numpy()[valid_mask]
for cid, row in zip(valid_context_ids, indices):
    geo_dist_predictions[cid] = [geo_item_ids[i] for i in row]
for cid in val_queries["context_id"][~valid_mask]:
    geo_dist_predictions[cid] = []

geo_dist_recall = recall_at_50(geo_dist_predictions, val_relevant)
print(f"Гео (top-50 по дистанции) Recall@50: {geo_dist_recall:.4f}")

Гео (top-50 по дистанции) Recall@50: 0.0933


Для сравнения - более простой вариант (exact location_id match + сортировка по популярности в train_part, без реальных координат) даёт **0.0750** - дистанция по факту точнее

## Локационно-ограниченный BM25

Глобальный BM25 (0.1861 соло) конкурирует по всему корпусу - популярные, но далёкие объявления. Сузим корпус до радиуса 20км от центроида локации и посчитаем BM25 только внутри этого подмножества через `weight_mask`.

In [7]:
import bm25s

from preprocessing import tokenize

items_corpus = train.drop_duplicates("item_id")[["item_id"]].reset_index(drop=True)
corpus_item_ids_bm25 = items_corpus["item_id"].tolist()
retriever = bm25s.BM25.load("../data/bm25s_index", load_corpus=False)

id_to_pos = {iid: i for i, iid in enumerate(corpus_item_ids_bm25)}
geo_pos_in_bm25 = np.array([id_to_pos[iid] for iid in geo_item_ids])
print("маппинг гео => bm25 построен:", len(geo_pos_in_bm25))

маппинг гео => bm25 построен: 344822


In [8]:
RADIUS_KM = 20
val_tokens = val_queries["search_query"].apply(tokenize).tolist()

geo_bm25_predictions = {}
for i, row in enumerate(val_queries.itertuples()):
    cid = row.context_id
    qloc = row.search_location_id
    if qloc not in loc_centroid.index:
        geo_bm25_predictions[cid] = []
        continue
    c_lat, c_lon = loc_centroid.loc[qloc, ["item_latitude", "item_longitude"]]
    idxs = geo_tree.query_radius(np.radians([[c_lat, c_lon]]), r=RADIUS_KM / 6371.0)[0]
    mask = np.zeros(len(corpus_item_ids_bm25), dtype=np.float32)
    mask[geo_pos_in_bm25[idxs]] = 1.0
    res = retriever.retrieve(
        [val_tokens[i]], corpus=corpus_item_ids_bm25, k=50, weight_mask=mask,
        show_progress=False, n_threads=0,
    )
    geo_bm25_predictions[cid] = list(res.documents[0])

geo_bm25_recall = recall_at_50(geo_bm25_predictions, val_relevant)
print(f"Локационно-ограниченный BM25 (радиус {RADIUS_KM}км) Recall@50: {geo_bm25_recall:.4f}")

Локационно-ограниченный BM25 (радиус 20км) Recall@50: 0.7346


Сравним с глобальным BM25 и с union обоих.

In [10]:
results_global = retriever.retrieve(val_tokens, corpus=corpus_item_ids_bm25, k=50, show_progress=False, n_threads=0)
global_bm25_predictions = {
    row.context_id: list(results_global.documents[i])
    for i, row in enumerate(val_queries.itertuples())
}
global_bm25_recall = recall_at_50(global_bm25_predictions, val_relevant)
print(f"Глобальный BM25 Recall@50: {global_bm25_recall:.4f}")

union_predictions = {
    cid: list(dict.fromkeys(
        list(geo_bm25_predictions.get(cid, [])) + list(global_bm25_predictions.get(cid, []))
    ))[:50]
    for cid in val_queries["context_id"]
}
union_recall = recall_at_50(union_predictions, val_relevant)
print(f"Union(глобальный, локационный BM25) Recall@50: {union_recall:.4f}")
print(f"прирост union над локационным соло: {union_recall - geo_bm25_recall:+.4f}")

Глобальный BM25 Recall@50: 0.1861
Union(глобальный, локационный BM25) Recall@50: 0.7702
прирост union над локационным соло: +0.0356


Порядок конкатенации важен - локационный первым, т.к. он сильнее; иначе global съедает все 50 слотов.\
 Хотя все еще это не очень интеллектуально :)